# Phases 0–3: the whole flow, end to end

*When Adaptive Posteriors Become Confidently Wrong* — quantile reliability in Bayesian
adaptive sensitivity experiments.

**The question.** A sequential experiment picks each stimulus $x$ using the current
posterior, then observes a binary response. We fit a probit
$p_\theta(x) = \Phi((x-\mu)/\sigma)$ and report credible intervals for response
quantiles. When are those intervals trustworthy — especially in the upper tail,
$q_{0.95}$ and $q_{0.99}$, which is where the decisions actually get made?

**How to read this notebook.** Phase 0 is computed live (it is fast). Phases 1–3 are
long runs — 53 to 67 minutes each — so those cells *load the summary CSVs the runners
wrote* rather than re-running anything. Nothing here is typed from memory; every number
is read from `results/summaries/`.

| Phase | Question | Answer |
|---|---|---|
| 0 | Does adaptive design break the likelihood? | No — exactly, under an ignorable policy |
| 1 | Does the baseline reproduce? | Yes for 5 of 6 designs |
| 2 | Is the machinery calibrated when the model is right? | Yes |
| 3 | What happens when the model is slightly wrong in the tail? | Coverage collapses to 0.43 |


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
SUM = REPO / "results" / "summaries"

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (7.5, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("repo:", REPO)
print("summaries found:", len(list(SUM.glob("*.csv"))))

---
## Phase 0 — the design does not corrupt the likelihood

The worry that motivates everything else: if stimulus $x_i$ was chosen using the
posterior built from $y_1..y_{i-1}$, the observations are not independent, so is
$\prod_i p_\theta(y_i \mid x_i)$ even the right likelihood?

It is. The full-history likelihood factors as

$$p_\theta(H_n) = \underbrace{\prod_i \pi(x_i \mid H_{i-1})}_{\text{design term}} \cdot \prod_i p_\theta(y_i \mid x_i)$$

and when the policy looks only at past data — *ignorable* — the design term does not
contain $\theta$, so it cancels out of the posterior. The check below computes the
difference of the two log-likelihoods across a 72-point $(\mu, \sigma)$ grid. If the
design term is free of $\theta$, that difference is **constant**, so its spread is 0.

In [ ]:
from src.factorization import (
    factorisation_residual,
    posterior_from_loglik,
    product_loglik,
    total_variation,
)
from src.policies import EntropyVectorPolicy, OraclePolicy, PolicyState
from src.priors import rotem_prior, rotem_stimulus_grid
from src.response_models import ProbitCurve
from src.rotem_particles import ParticlePosterior

prior = rotem_prior("well", "independent")
candidates = rotem_stimulus_grid("well", n_points=60)
curve = ProbitCurve(30.0, 3.0)
N_PARTICLES, N_STEPS = 800, 12

# the theta grid the residual is evaluated on
MU, SG = np.meshgrid(np.linspace(26.0, 34.0, 9), np.linspace(1.5, 5.0, 8), indexing="ij")
MU, SG = MU.ravel(), SG.ravel()


def simulate(policy, seed):
    """Run one short sequential experiment and return its (x, y) history."""
    rng = np.random.default_rng(seed)
    posterior = ParticlePosterior(prior, N_PARTICLES, rng)
    state = PolicyState(candidates=candidates, posterior=posterior)
    xs, ys = [], []
    u = np.random.default_rng(seed + 5000).random(N_STEPS)
    for k in range(N_STEPS):
        x, _ = policy.select(state, rng)
        y = int(u[k] < float(np.atleast_1d(curve.prob(np.array([x])))[0]))
        posterior.update(x, y)
        xs.append(x)
        ys.append(y)
        state.x_hist, state.y_hist, state.step = xs, ys, k + 1
        state.invalidate()
    return np.asarray(xs), np.asarray(ys)


def check_invariant(policy_factory, seed):
    """Spread of the design term over theta, and the resulting posterior difference."""
    x, y = simulate(policy_factory(), seed)
    resid = factorisation_residual(
        policy_factory(), prior, candidates, x, y, MU, SG,
        n_particles=N_PARTICLES, seed=seed,
    )
    finite = np.isfinite(resid)
    log_prior = prior.logpdf(MU, SG)
    prod = product_loglik(x, y, MU, SG)
    tv = total_variation(
        posterior_from_loglik(prod, log_prior),
        posterior_from_loglik(prod + resid, log_prior),
    )
    return float(np.ptp(resid[finite])), float(tv)


spread_ad, tv_ad = check_invariant(lambda: EntropyVectorPolicy(), seed=11)
spread_or, tv_or = check_invariant(lambda: OraclePolicy(curve, p=0.5, sharpness=2.0), seed=37)

pd.DataFrame({
    "design": ["entropy_vector (ignorable)", "oracle (NON-ignorable)"],
    "residual spread (nats)": [spread_ad, spread_or],
    "posterior total variation": [tv_ad, tv_or],
})

**What this shows.** The adaptive design gives spread 0 and total variation 0: using the
simple product likelihood and using the full history likelihood give *the same posterior*.
Adaptive dependence is not an approximation here, it is exact.

The second row is the point of the exercise. The oracle policy peeks at the true
$\theta$ when choosing $x$ — it is deliberately non-ignorable — and it breaks the
invariant hard (≈84 nats, TV 0.77). Without that negative control, an implementation
that simply *forgot* the design term would pass the first row vacuously. This is why
`CLAUDE.md` treats "the negative control must keep failing" as an invariant.

One consequence worth stating: this closes the question. The likelihood is **not** an
independence approximation, so the failures found later in Phase 3 cannot be blamed on it.

---
## Phase 1 — does the published baseline reproduce?

Before stress-testing a method you have to be running the same method. Phase 1 re-runs
Rotem's baseline across settings, designs and targets, 250 replicates per cell, and
compares MSE against the published tables. "Agreement" means within 2 combined standard
errors — stated, not eyeballed.

In [ ]:
vs_pub = pd.read_csv(SUM / "phase1_vs_published.csv")

by_policy = (
    vs_pub.groupby("policy")
    .agg(cells=("within_2se", "size"),
         within_2se=("within_2se", "sum"),
         max_abs_z=("z_combined", lambda s: s.abs().max()))
    .sort_values("max_abs_z")
)
by_policy["agree"] = (by_policy.within_2se / by_policy.cells).map("{:.0%}".format)
by_policy.round(2)

In [ ]:
ok = vs_pub[vs_pub.policy != "entropy_sigma"]
fig, ax = plt.subplots()
ax.scatter(ok.mse_published, ok.mse_reproduced, s=22, label="5 reproducing designs")
bad = vs_pub[vs_pub.policy == "entropy_sigma"]
ax.scatter(bad.mse_published, bad.mse_reproduced, s=22, marker="x",
           color="crimson", label="entropy_sigma (D13)")
lim = [0, ok[["mse_published", "mse_reproduced"]].to_numpy().max() * 1.05]
ax.plot(lim, lim, "k--", lw=1, label="exact agreement")
ax.set(xlim=lim, ylim=lim, xlabel="published MSE", ylabel="reproduced MSE",
       title="Phase 1: reproduction against the published tables")
ax.legend(fontsize=8)
plt.show()

print(f"all cells:              {vs_pub.within_2se.sum()}/{len(vs_pub)} within 2 SE")
print(f"excluding entropy_sigma: {ok.within_2se.sum()}/{len(ok)} within 2 SE, "
      f"max |z| = {ok.z_combined.abs().max():.2f}")

**What this shows.** Five of six designs reproduce (claim C2). The plot is clipped to
the reproducing range so the diagonal is readable — `entropy_sigma` is off the chart, not
off by a little: its max $|z|$ is over 1000, so it is a structural disagreement rather
than noise. The criterion was verified exact against a closed form, which means the fault
is in the comparison, not the implementation — it stays **Unresolved** (C2b, `D13`)
rather than being quietly dropped or explained away.

`bruceton` is the weakest of the five that do reproduce (12/20 cells, max $|z|$ 3.38).
Worth knowing before leaning on it.

---
## Phase 2 — when the model is correct, is the machinery calibrated?

This is the control condition, and it has to be established before any failure in Phase 3
means anything. Data are generated from a **true probit** and fitted with a probit, so the
model is exactly right. Two questions:

1. **SBC** — is the posterior calibrated *with the adaptive policy in the loop*?
2. **Computational fidelity** — how close is the fast particle posterior to the slow
   reference grid posterior?

Note the `backend` column below: `ref` is the reference grid posterior, `particle` is
Rotem's weighted-particle one. Keeping them apart is the whole point — computational
error and model misspecification are different chapters and must not be blamed on each
other.

In [ ]:
sbc = pd.read_csv(SUM / "phase2_sbc_uniformity.csv")
print("SBC rank-uniformity tests passing at the 0.05 level:")
print(sbc.groupby("backend").passes_ks_05.agg(["sum", "size"]))
print("\nany failures:")
print(sbc.loc[~sbc.passes_ks_05, ["backend", "policy", "target", "ks_p", "coverage"]]
      .to_string(index=False))

sbc[sbc.backend == "ref"][["policy", "target", "n", "ks_p", "coverage", "coverage_se"]].round(4)

**What this shows.** The reference posterior passes all 10 uniformity tests, under both a
non-adaptive (`uniform_grid`) and an adaptive (`entropy_vector`) design, on every target
including the tail quantiles. Coverage sits at nominal 0.95 throughout. That is claim C3,
and it is what licenses reading any later coverage loss as a real effect.

The particle backend fails exactly one test of 10 (`uniform_grid`, $q_{0.95}$,
$p = 0.024$). One failure in ten at the 0.05 level is about what chance delivers, so this
is not evidence of a broken sampler — but it is the same direction as the next cell, and
it is reported rather than rounded away.

In [ ]:
fidelity = pd.read_csv(SUM / "phase2_computational_fidelity.csv")
fidelity[["target", "shift_in_ref_sd_units", "sd_ratio_mean", "sd_ratio_p05", "sd_ratio_p95"]].round(4)

**What this shows.** The particle posterior sits about 1% of a reference standard
deviation away from the reference posterior, and is about 0.3% narrower. Close — but the
gap is measurable and consistently in the *narrow* direction (claim C4). That makes the
particle approximation a chapter of the thesis rather than a defect to hide, and it is
why Phase 3 reports the reference backend when a claim has to carry weight.

---
## Phase 3 — the result: confident and wrong in the tail

Now break the model, but only slightly, and only where it is hard to see. The
`TailPerturbedCurve` is **exactly probit below the 0.85 quantile** — the region the
adaptive design actually visits — and diverges only above it, near $q_{0.95}$ and
$q_{0.99}$. The likelihood sees almost nothing wrong.

This phase was **pre-registered** (`manuscript/phase3_protocol.md`, frozen before any
Phase 3 experiment ran), and the three arms are common-random-number paired:

| Arm | Curve | Design | Role |
|---|---|---|---|
| **A** | tail-perturbed | adaptive | the suspect |
| **B** | tail-perturbed | fixed | same misspecification, no adaptivity |
| **C** | probit | adaptive | same adaptivity, no misspecification |

A vs C isolates misspecification; A vs B isolates what the *adaptive design* adds.

In [ ]:
confirm = pd.read_csv(SUM / "phase3b_confirm_summary.csv")
arm_label = {"A": "A  tail + adaptive", "B": "B  tail + fixed", "C": "C  probit + adaptive"}
confirm["arm"] = confirm.label.map(arm_label)

coverage = confirm.pivot_table(index="arm", columns="target", values="coverage")
print("Credible interval coverage (nominal 0.95, 300 replicates per cell):")
display(coverage.round(3))

fig, ax = plt.subplots()
coverage.T.plot.bar(ax=ax, rot=0, width=0.78)
ax.axhline(0.95, color="k", ls="--", lw=1)
ax.text(2.35, 0.955, "nominal 0.95", fontsize=8, va="bottom", ha="right")
ax.set(ylim=(0, 1.05), ylabel="coverage", xlabel="target quantile",
       title="Phase 3 confirmatory: coverage collapses in the tail, only when adaptive")
ax.legend(fontsize=8, loc="lower left")
plt.show()

**What this shows — the strongest result in the project (C5).**

Read the bars left to right. At the median everything is fine: all three arms are near
0.95, so nothing is obviously broken. Move into the tail and arm A falls to **0.63** at
$q_{0.95}$ and **0.43** at $q_{0.99}$ — a nominally 95% interval that contains the truth
less than half the time.

The two controls are what make this interpretable:

- **C** (probit + adaptive) stays at 0.97 everywhere → adaptivity alone is harmless.
- **B** (tail + fixed) drops only to 0.63 at $q_{0.99}$ → misspecification alone costs
  something, but much less.

So neither ingredient is sufficient on its own. The damage comes from the *interaction*:
the adaptive design concentrates sampling where the model is right, which is precisely
what stops it from ever learning that the tail is wrong.

In [ ]:
fc = confirm.pivot_table(index="arm", columns="target", values="false_certainty")
print("False certainty — interval both unusually NARROW and wrong:")
display(fc.round(3))

h2 = pd.read_csv(SUM / "phase3b_confirm_h2.csv")
print("\nPre-registered H2, CRN-paired adaptive vs fixed:")
h2.round(4)

**Why this is worse than plain undercoverage.** A wide interval that misses is honest —
it says it was unsure. *False certainty* counts runs where the interval was unusually
narrow **and** missed: the method reports high confidence exactly when it is wrong. Arm A
does this on 25% of runs in the tail, against 1–2% for the correctly-specified arm C.

The paired test is the formal version. Same random numbers, adaptive vs fixed: the
adaptive design adds **+0.103** (SE 0.032) false certainty at $q_{0.95}$ and **+0.147**
(SE 0.030) at $q_{0.99}$ — several SEs clear of zero, and the pre-registered H2 threshold
is met on both.

In [ ]:
screen = pd.read_csv(SUM / "phase3b_screen_summary.csv")
atlas = screen[(screen.block == "I") & (screen.backend == "ref") & (screen.n == 50)
               & (screen.pol == "entropy_vector") & (screen.target == "q0.99")]
atlas = atlas.sort_values("coverage")

fig, ax = plt.subplots()
ax.barh(atlas.dgp, atlas.coverage, xerr=atlas.coverage_se,
        color=["crimson" if c < 0.8 else "steelblue" for c in atlas.coverage])
ax.axvline(0.95, color="k", ls="--", lw=1)
ax.set(xlabel="$q_{0.99}$ coverage", xlim=(0, 1.05),
       title="Failure atlas: which wrong curves actually hurt (screening, 30 reps)")
plt.show()

atlas[["dgp", "n_rep", "coverage", "coverage_se"]].round(3).to_string(index=False)

**What this shows.** Most ways of being wrong are harmless. Logistic, cloglog, mixture
and Beta-CDF alternatives all sit at or above nominal — a globally mis-shaped curve
inflates the posterior enough to absorb its own bias. Only two families fail: the
constructed tail perturbation (0.40) and `robit` (0.50), which shows the same signature
at roughly half the size (C6).

**Read the error bars.** These are 30 replicates, SE ≈ 0.09 — *screening* precision, not
confirmatory. This panel is a map of where to look next, not a result to quote. Only the
300-replicate confirmatory arms above carry that weight, and the difference in tier is
tracked per claim in `docs/claims.md`.

In [ ]:
attr = pd.read_csv(SUM / "phase3a_attribution.csv")
tail = attr[(attr.target == "q0.99") & (attr.sigma_true == 0.3)]
tail[["arm", "cov_particle", "cov_reference", "cov_diff", "cov_diff_se", "ess"]].round(3)

**A separate, computational failure (C8).** In a narrow corner ($\sigma = 0.3$) the
reference posterior holds coverage at 0.94 while the particle posterior drops to 0.47 —
so this one is *not* misspecification, it is the approximation. The surprise is the
ordering: turning rejuvenation **off** does better (0.87) than Rotem's rejuvenation rule
(0.47) despite roughly 30× lower effective sample size, and lowering the KL threshold to
0.02 recovers 0.88. Throwing 16× more particles at it only reaches 0.69.

Keeping this in its own box is the discipline the project runs on: a computational defect
must never be reported as evidence about misspecification. Note the frequency — about
1.6% of runs, and zero for $\sigma \ge 1.2$.

---
## Where this leaves things

The chain: the likelihood is sound (Phase 0) → the implementation matches the published
baseline (Phase 1) → the machinery is calibrated when the model is right (Phase 2) →
therefore the tail collapse in Phase 3 is a real property of the method, not an artefact
of any of the three.

**The mechanism.** Mutual-information stimulus selection concentrates sampling where the
model already fits. Tail quantiles are then extrapolations from a region that contains no
evidence against the model — and the posterior is *narrow* there, because a
well-fitting region looks like strong information. Confident, and wrong.

**What is not established.**

- C5 is one constructed curve family at one horizon. That is the main exposure, and the
  screening atlas is suggestive of generality, not proof of it.
- The exploratory control does **not** restore coverage (C10, Negative) — so a safeguard
  is genuine research for Phase 6, not a write-up task.
- Phases 4–7 (theory, the online reliability map, safeguards, extensions) are not started.
- Every result above predates the provenance contract: all 13 manifests are unpinned and
  carry no `run_id` (`D19`, `D20`). The numbers are checkable against the files, but not
  yet tied to a state of the source. A clean re-run is what closes that.

Authoritative status for every claim is `docs/claims.md`; deviations are in
`docs/discrepancies.md`.